In [0]:
from pyspark.sql.functions import current_timestamp
df = spark.read.csv("/Volumes/workspace/bronze/emp",emptyValue=None, header=True)\
                    .withColumn("L1", current_timestamp())
display(df)

Emp_no,EmpName,EmpAdd,DeptName,Gender,L0_upd,L1
1,Rajesh,Noida,HR,M,09-11-2026,2026-11-02T17:25:38.949Z
2,John,Vizag,HR,M,10-11-2026,2026-11-02T17:25:38.949Z
3,Joanne,Pune,IT,F,04-11-2026,2026-11-02T17:25:38.949Z
4,Joanne,Hyd,IT,F,09-11-2026,2026-11-02T17:25:38.949Z
5,Vaibhav,Lucknow,HR,M,06-11-2026,2026-11-02T17:25:38.949Z
6,Smith,null,IT,F,07-11-2026,2026-11-02T17:25:38.949Z
7,Sam,kkd,IT,F,08-11-2026,2026-11-02T17:25:38.949Z
8,Sam,kkd,IT,F,09-11-2026,2026-11-02T17:25:38.949Z
9,Sam,kkd,IT,F,09-11-2026,2026-11-02T17:25:38.949Z
10,Abhi,America,Manager,M,10-11-2026,2026-11-02T17:25:38.949Z


In [0]:
from pyspark.sql.functions import max
silver_emp = spark.read.table("workspace.bronze.silver_emp")
max_date = silver_emp.select(max("L0_upd")).collect()[0][0]
display(max_date) 

'09-11-2026'

In [0]:
sour_max_rec = df.filter(df.L0_upd > max_date)
sour_max_rec.display()

Emp_no,EmpName,EmpAdd,DeptName,Gender,L0_upd,L1
2,John,Vizag,HR,M,10-11-2026,2026-11-02T17:25:52.038Z
10,Abhi,America,Manager,M,10-11-2026,2026-11-02T17:25:52.038Z


In [0]:
from delta.tables import DeltaTable

silver_tbl = DeltaTable.forName(spark, "workspace.bronze.silver_emp")
silver_tbl.alias("target").merge(sour_max_rec.alias("source"), "target.Emp_no = source.Emp_no")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from workspace.bronze.silver_emp

Emp_no,EmpName,EmpAdd,DeptName,Gender,L0_upd,L1
3,Joanne,Pune,IT,F,04-11-2026,2026-11-02T16:48:03.931Z
5,Vaibhav,Lucknow,HR,M,06-11-2026,2026-11-02T16:48:03.931Z
6,Smith,null,IT,F,07-11-2026,2026-11-02T16:48:03.931Z
7,Sam,kkd,IT,F,08-11-2026,2026-11-02T16:48:03.931Z
1,Rajesh,Noida,HR,M,09-11-2026,2026-11-02T17:18:49.217Z
4,Joanne,Hyd,IT,F,09-11-2026,2026-11-02T17:18:49.217Z
8,Sam,kkd,IT,F,09-11-2026,2026-11-02T17:18:49.217Z
9,Sam,kkd,IT,F,09-11-2026,2026-11-02T17:18:49.217Z
2,John,Vizag,HR,M,10-11-2026,2026-11-02T17:25:59.590Z
10,Abhi,America,Manager,M,10-11-2026,2026-11-02T17:25:59.590Z


In [0]:
12